In [ ]:

import numpy as np
import torch
import torch.nn as nn
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Fijar semilla general
torch.manual_seed(42)

# ==========================================
# 1. Datos y Escalado
# ==========================================
iris = load_iris()
X, y = iris.data, iris.target
nombres_clases = iris.target_names  # ['setosa', 'versicolor', 'virginica']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

escalador = StandardScaler()
X_train_esc = escalador.fit_transform(X_train)
X_test_esc = escalador.transform(X_test)

X_train_t = torch.tensor(X_train_esc, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)


# ==========================================
# 2. Definición del Modelo
# ==========================================
class RedIris(nn.Module):
    def __init__(self):
        super().__init__()
        self.capas = nn.Sequential(
            nn.Linear(4, 16),
            nn.ReLU(),
            nn.Linear(16, 3),
        )

    def forward(self, x):
        return self.capas(x)


modelo = RedIris()
criterio = nn.CrossEntropyLoss()
optimizador = torch.optim.Adam(modelo.parameters(), lr=0.01)

# ==========================================
# 3. Entrenamiento rápido (200 épocas) y guardado
# ==========================================
modelo.train()
for epoca in range(200):
    salida = modelo(X_train_t)
    loss = criterio(salida, y_train_t)
    optimizador.zero_grad()
    loss.backward()
    optimizador.step()

# Guarda el archivo en la carpeta actual del notebook
torch.save(modelo.state_dict(), "red_iris.pth")
print("Modelo entrenado y archivo 'red_iris.pth' creado correctamente.\n")

# ==========================================
# 4. Carga de los pesos (cumpliendo el reto)
# ==========================================
modelo_eval = RedIris()
modelo_eval.load_state_dict(torch.load("red_iris.pth", weights_only=True))
modelo_eval.eval()

# ==========================================
# 5. Definir 3 flores inventadas y predecir
# ==========================================
flores_inventadas = [
    [5.0, 3.4, 1.5, 0.2],  # Medidas de Setosa
    [6.0, 2.8, 4.3, 1.3],  # Medidas de Versicolor
    [7.5, 3.2, 6.5, 2.2],  # Medidas de Virginica
]

flores_esc = escalador.transform(flores_inventadas)
flores_tensor = torch.tensor(flores_esc, dtype=torch.float32)

with torch.no_grad():
    salidas = modelo_eval(flores_tensor)
    probabilidades = torch.softmax(salidas, dim=1)
    predicciones = salidas.argmax(dim=1)

# ==========================================
# 6. Salida de resultados
# ==========================================
print(" PREDICCIONES DEL MODELO CARGADO ")
for i, (flor, pred_idx, probs) in enumerate(
    zip(flores_inventadas, predicciones, probabilidades), start=1
):
    clase = nombres_clases[pred_idx.item()]
    confianza = probs[pred_idx].item() * 100
    print(f"Flor {i}: {flor}")
    print(f"  -> Especie: {clase.upper()} (Clase {pred_idx.item()})")
    print(f"  -> Certeza: {confianza:.2f}%\n")

Modelo entrenado y archivo 'red_iris.pth' creado correctamente.

=== PREDICCIONES DEL MODELO CARGADO ===
Flor 1: [5.0, 3.4, 1.5, 0.2]
  -> Especie: SETOSA (Clase 0)
  -> Certeza: 99.98%

Flor 2: [6.0, 2.8, 4.3, 1.3]
  -> Especie: VERSICOLOR (Clase 1)
  -> Certeza: 99.90%

Flor 3: [7.5, 3.2, 6.5, 2.2]
  -> Especie: VIRGINICA (Clase 2)
  -> Certeza: 100.00%

